# Model Evaluation Activity

Evaluating a machine learning model is a fundamental process. In particular, analyzing the degree of bias and variance is essential for understanding its performance and generalization capacity.

In this notebook, we will work with the Diabetes dataset available in scikit-learn, on which we will fit different regression models. Afterwards, we will build learning curves and validation curves in order to obtain information about the level of bias and variance present in each model.

In addition, you will be asked to answer some questions and complete code fragments, which will help reinforce the concepts covered and consolidate your understanding of the model evaluation process.

# [Diabetes dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset)

Diabetes dataset
442 samples (rows), 10 features (columns)

Features:
- age – Age of the patient
- sex – Sex of the patient
- bmi – Body mass index.
- bp – Average blood pressure.
- s1 – tc, total serum cholesterol.
- s2 – ldl, low-density lipoproteins.
- s3 – hdl, high-density lipoproteins.
- s4 – tch, total cholesterol / HDL.
- s5 – ltg, possibly log of serum triglycerides level.
- s6 – glu, blood sugar level.

Target Variable:

A quantitative measure of diabetes progression one year after the first observation..

### SGDRegressor

As a first example, we are going to create a learning curve for a regressor from the SGDRegressor class.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.linear_model import SGDRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Data
X, y = load_diabetes(return_X_y=True)

# We check the mean and standard deviation of each feature
print("Mean per feature:")
print(np.mean(X, axis=0))
print("\nStandard deviation per feature:")
print(np.std(X, axis=0))


# Real model: pipeline with scaling + SGD
model = make_pipeline(
    StandardScaler(),             # So that the standard deviation is 1
    SGDRegressor(
        loss="squared_error",     # equivalent to MSE
        learning_rate="constant",
        eta0=1e-3,                # initial learning rate
        max_iter=2000,            # more iterations to ensure convergence
        tol=1e-3,
        random_state=42,
    )
)

# Learning curve for the model
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Means and standard deviations
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Baselines ========

# DummyRegressor with mean
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean_scores = cross_val_score(dummy_mean, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_mean = -np.mean(dummy_mean_scores)

# DummyRegressor with median
dummy_median = DummyRegressor(strategy="median")
dummy_median_scores = cross_val_score(dummy_median, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_median = -np.mean(dummy_median_scores)

# ======== Plot ========

plt.plot(train_sizes, train_mean, label="Training (SGDRegressor)", color="blue")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(train_sizes, val_mean, label="Validation (SGDRegressor)", color="orange")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

# Horizontal baseline lines
plt.axhline(y=baseline_mean, color="red", linestyle="--", label="Dummy (mean)")
plt.axhline(y=baseline_median, color="green", linestyle="--", label="Dummy (median)")

plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning curve with SGDRegressor and baselines")
plt.ylim(0, 7000) 
plt.legend()
plt.grid()
plt.show()

Edit this cell to answer the following questions:

1. Why do we use cross-validation to generate the learning curve?
2. Would the model’s performance with these parameters benefit from increasing the number of training examples?
3. What degree of bias does the model show at the beginning and at the end of training?
4. What degree of variance does the model show at the beginning and at the end of training?



### Polinomial regression

Modify the previous code to include second-order terms and interactions between the variables, or features. You can copy the code from the previous cell and modify only the necessary part. This will allow you to compare both plots.

To do this, you can use the PolynomialFeatures function available in sklearn.preprocessing.

In [ ]:
# Your code here

Answer the following questions:

1. Did the model’s performance improve?
2. What degree of bias does the model show at the beginning and at the end of training?
3. What degree of variance does the model show at the beginning and at the end of training?

#### Regularization

To demonstrate how regularization affects the degree of bias and variance, add regularization to the SGDRegressor in the following cell. I recommend starting with Ridge-type regularization. Check the documentation for the SGDRegressor class to obtain information about the penalty and alpha arguments, which will allow you to define the type of regularization and its intensity.

Try different values of alpha. For example, use L2 regularization (penalty="l2") and experiment with alpha values such as 0.1, 1, 10, and 100.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.linear_model import SGDRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Data
X, y = load_diabetes(return_X_y=True)

# ===== Model with polynomials + Ridge regularization =====
model_poly = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=True),
    StandardScaler(),
    SGDRegressor(
        loss="squared_error",     
        learning_rate="constant",
        eta0=1e-3,                
        max_iter=2000,            
        tol=1e-3,
        random_state=42,
        # Add Ridge regularization here
    )
)

# Learning curve with polynomials
train_sizes, train_scores, val_scores = learning_curve(
    model_poly, X, y, cv=5, scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Means and standard deviations
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Baselines ========
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean_scores = cross_val_score(dummy_mean, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_mean = -np.mean(dummy_mean_scores)

dummy_median = DummyRegressor(strategy="median")
dummy_median_scores = cross_val_score(dummy_median, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_median = -np.mean(dummy_median_scores)

# ======== Plot ========
plt.plot(train_sizes, train_mean, label="Training (SGD + Poly + Ridge)", color="blue")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(train_sizes, val_mean, label="Validation (SGD + Poly + Ridge)", color="orange")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

plt.axhline(y=baseline_mean, color="red", linestyle="--", label="Dummy (mean)")
plt.axhline(y=baseline_median, color="green", linestyle="--", label="Dummy (median)")

plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning Curve with Ridge (SGDRegressor)")
plt.ylim(0, 7000)
plt.legend()
plt.grid()
plt.show()

Edit this cell to answer the following questions:

1. What degree of bias and variance do you observe with small values of alpha, for example, 0.1?
2. What degree of bias and variance do you observe with large values of alpha, for example, 100?
3. How would you explain this difference in your own words?


### Regression with K-Nearest Neighbors

1. Replace the SGDRegressor class with KNeighborsRegressor. Start with the following parameters:

- n_neighbors=5,     
- weights="uniform", 
- metric="minkowski", 
- p=2

2. Check the documentation to understand the function of the weights and metric parameters.
3. Generate the corresponding validation curve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Data
X, y = load_diabetes(return_X_y=True)

# Real model: pipeline with scaling + KNN
model = make_pipeline(
    StandardScaler(),
    # Define your instantiation of the KNeighborsRegressor class here
)

# Learning curve for the model
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Means and standard deviations
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Baselines ========

# DummyRegressor with mean
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean_scores = cross_val_score(dummy_mean, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_mean = -np.mean(dummy_mean_scores)

# DummyRegressor with median
dummy_median = DummyRegressor(strategy="median")
dummy_median_scores = cross_val_score(dummy_median, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_median = -np.mean(dummy_median_scores)

# ======== Plot ========

plt.plot(train_sizes, train_mean, label="Training (KNN)", color="blue")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(train_sizes, val_mean, label="Validation (KNN)", color="orange")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

# Horizontal baseline lines
plt.axhline(y=baseline_mean, color="red", linestyle="--", label="Dummy (mean)")
plt.axhline(y=baseline_median, color="green", linestyle="--", label="Dummy (median)")

plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning Curve with KNN and Baselines")
plt.ylim(0, 7000)  # Adjust the y-axis limit for better visualization
plt.grid()
plt.legend()
plt.show()

Edit this cell and answer the following questions:

1. Did the model’s performance improve compared with the previous model, SGDRegressor?
2. What degree of bias does the model show at the beginning and at the end of training?
3. What degree of variance does the model show at the beginning and at the end of training?
4. Which hyperparameter do you think we should adjust to try to improve the model?

The following cell creates a validation curve in which the horizontal axis shows different values of the parameter k, and the vertical axis shows the performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import validation_curve
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Data
X, y = load_diabetes(return_X_y=True)

# KNN model inside a pipeline with scaling
model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor()
)

# Range of neighbors to test
param_range = np.arange(1, 31)

# Validation curve
metric = 'neg_mean_squared_error'  # evaluation metric
train_scores, val_scores = validation_curve(
    model, X, y,
    param_name="kneighborsregressor__n_neighbors",  # full name in the pipeline
    param_range=param_range,
    cv=5,
    scoring=metric,
    n_jobs=-1
)

# Convert from negative MSE to positive MSE
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# Plot
plt.plot(param_range, train_mean, label="Training", color="blue")
plt.fill_between(param_range, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(param_range, val_mean, label="Validation", color="orange")
plt.fill_between(param_range, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

plt.xlabel("Number of neighbors (k)")
plt.ylabel("MSE")
plt.title("Validation Curve - KNeighborsRegressor")
plt.legend()
plt.grid()
plt.show()

Answer the following questions:

1. What degree of bias and variance is observed for small values of k?
2. What degree of bias and variance is observed for large values of k?
3. Which value of k do you consider most appropriate, and why?

#### Hyperparameter tuning

1. Implement a grid search using the GridSearchCV function from scikit-learn to find the optimal value of the parameter k.
2. Split the dataset into training and test sets.
3. Perform the hyperparameter search using only the training set.
4. Train the model using the optimal value of k obtained from the search.
5. Evaluate the model on the test set and report the generalization error.

In [ ]:
# Your code here

Edit the following cells to answer these questions:

1. What is the optimal value of k according to the grid search?
2. Why is it important to use cross-validation to find the optimal value of k?

# Extra Examples - Using Other Types of Regressors

#### Regressor SVR

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Data
X, y = load_diabetes(return_X_y=True)

# Real model: pipeline with scaling + SVR
model = make_pipeline(
    StandardScaler(),
    SVR(
        kernel="rbf",   # RBF kernel (Gaussian); you can try "linear" or "poly"
        C=1.0,          # regularization parameter
        epsilon=0.1     # tolerance margin in regression
    )
)

# Learning curve for the model
train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Means and standard deviations
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Baselines ========

# DummyRegressor with mean
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean_scores = cross_val_score(dummy_mean, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_mean = -np.mean(dummy_mean_scores)

# DummyRegressor with median
dummy_median = DummyRegressor(strategy="median")
dummy_median_scores = cross_val_score(dummy_median, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_median = -np.mean(dummy_median_scores)

# ======== Plot ========

plt.plot(train_sizes, train_mean, label="Training (SVR)", color="blue")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(train_sizes, val_mean, label="Validation (SVR)", color="orange")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

# Horizontal baseline lines
plt.axhline(y=baseline_mean, color="red", linestyle="--", label="Dummy (mean)")
plt.axhline(y=baseline_median, color="green", linestyle="--", label="Dummy (median)")

plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning Curve with SVR and Baselines")
plt.legend()
plt.show()

Validation curve for the C parameter of the SVR.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import validation_curve
from sklearn.svm import SVR

# Data
X, y = load_diabetes(return_X_y=True)

# Model: SVR with a linear kernel for simplicity
model = SVR(kernel="linear")

# Range of C values to test
param_range = np.logspace(-3, 5, 10)  # from 0.001 to 1000

# Validation curve with MSE
train_scores, val_scores = validation_curve(
    model, X, y,
    param_name="C",
    param_range=param_range,
    cv=5,
    scoring="neg_mean_squared_error",  # we use MSE
    n_jobs=-1
)

# Convert the values to positive
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Plot ========
plt.semilogx(param_range, train_mean, label="Training (MSE)", color="blue")
plt.fill_between(param_range, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.semilogx(param_range, val_mean, label="Validation (MSE)", color="orange")
plt.fill_between(param_range, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

plt.xlabel("C")
plt.ylabel("MSE")
plt.title("Validation Curve for SVR with Linear Kernel")
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

### Regresor HistGradientBoostingRegressor

Example learning curve for a regressor of the HistGradientBoostingRegressor type.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import learning_curve, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor

# Data
X, y = load_diabetes(return_X_y=True)

# ===== Model with HistGradientBoosting =====
model_hgb = HistGradientBoostingRegressor(
    max_depth=5,        # maximum depth of the trees; try 5 or 2
    learning_rate=0.1,  # learning rate
    max_iter=200,       # number of iterations, or trees; try 200 or 100
    random_state=0,
    min_samples_leaf=30
)

# Learning curve
train_sizes, train_scores, val_scores = learning_curve(
    model_hgb, X, y, cv=5, scoring="neg_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Means and standard deviations
train_mean = -np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = -np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

# ======== Baselines ========
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean_scores = cross_val_score(dummy_mean, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_mean = -np.mean(dummy_mean_scores)

dummy_median = DummyRegressor(strategy="median")
dummy_median_scores = cross_val_score(dummy_median, X, y, cv=5, scoring="neg_mean_squared_error")
baseline_median = -np.mean(dummy_median_scores)

# ======== Plot ========
plt.plot(train_sizes, train_mean, label="Training (HistGBR)", color="blue")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.2, color="blue")

plt.plot(train_sizes, val_mean, label="Validation (HistGBR)", color="orange")
plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.2, color="orange")

plt.axhline(y=baseline_mean, color="red", linestyle="--", label="Dummy (mean)")
plt.axhline(y=baseline_median, color="green", linestyle="--", label="Dummy (median)")

plt.xlabel("Training set size")
plt.ylabel("MSE")
plt.title("Learning Curve with HistGradientBoostingRegressor")
plt.ylim(0, 7000)
plt.legend()
plt.grid()
plt.show()